## **Clean HMDA Loan Data**

In [ ]:
import pandas as pd
import numpy as np
df = pd.read_csv("../A. Data Pipeline/Data/bronze/hmda_loans_raw.csv", low_memory=False)
print(f"Raw shape: {df.shape}")

Raw shape: (2830687, 100)


In [38]:
df = df[df["action_taken"].isin([1, 3])].copy()
df["is_denied"] = (df["action_taken"] == 3).astype(int)
print(f"Filtered: {df.shape}")
print(df["is_denied"].value_counts())

Filtered: (2830687, 101)
is_denied
0    2099761
1     730926
Name: count, dtype: int64


In [39]:
KEEP_COLS = [
    "is_denied",
    "loan_amount", 
    "loan_to_value_ratio",
    "interest_rate", 
    "loan_term", 
    "loan_type", 
    "loan_purpose",
    "income", 
    "debt_to_income_ratio",
    "applicant_age", 
    "applicant_sex", 
    "applicant_race-1",
    "derived_dwelling_category", 
    "occupancy_type",
    "construction_method", 
    "total_units", 
    "property_value",
    "state_code", 
    "county_code", 
    "lei", 
    "activity_year",
    "denial_reason-1", 
    "denial_reason-2", 
    "denial_reason-3",
]

KEEP_COLS = [c for c in KEEP_COLS if c in df.columns]
df = df[KEEP_COLS].copy()
print(f"Selected: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

Selected: (2830687, 24)
Columns: ['is_denied', 'loan_amount', 'loan_to_value_ratio', 'interest_rate', 'loan_term', 'loan_type', 'loan_purpose', 'income', 'debt_to_income_ratio', 'applicant_age', 'applicant_sex', 'applicant_race-1', 'derived_dwelling_category', 'occupancy_type', 'construction_method', 'total_units', 'property_value', 'state_code', 'county_code', 'lei', 'activity_year', 'denial_reason-1', 'denial_reason-2', 'denial_reason-3']


In [40]:
numeric_cols = [
    "loan_amount", "income", "interest_rate", "loan_term",
    "loan_to_value_ratio", "debt_to_income_ratio", "property_value",
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

print(df.dtypes)

is_denied                      int64
loan_amount                  float64
loan_to_value_ratio          float64
interest_rate                float64
loan_term                    float64
loan_type                      int64
loan_purpose                   int64
income                       float64
debt_to_income_ratio         float64
applicant_age                    str
applicant_sex                  int64
applicant_race-1             float64
derived_dwelling_category        str
occupancy_type                 int64
construction_method            int64
total_units                      str
property_value               float64
state_code                       str
county_code                  float64
lei                              str
activity_year                  int64
denial_reason-1                int64
denial_reason-2              float64
denial_reason-3              float64
dtype: object


In [41]:
missing_pct = df.isnull().mean()
print(f"Columns >50% missing: {missing_pct[missing_pct > 0.5].index.tolist()}")

df = df.loc[:, df.isnull().mean() < 0.6]

num_cols = df.select_dtypes(include=[np.number]).columns.drop("is_denied", errors="ignore")
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

cat_cols = df.select_dtypes(include=["object"]).columns
for col in cat_cols:
    df[col] = df[col].fillna("Unknown")
    
print(f"After cleaning: {df.shape}")
print(f"Remaining nulls: {df.isnull().sum().sum()}")

Columns >50% missing: ['debt_to_income_ratio', 'denial_reason-2', 'denial_reason-3']


C:\Users\KhaiH\AppData\Local\Temp\ipykernel_29028\1768563580.py:9: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include=["object"]).columns


After cleaning: (2830687, 22)
Remaining nulls: 0


In [42]:
df["loan_to_income"] = df["loan_amount"] / (df["income"].replace(0, np.nan) + 1)
df["high_dti"] = (df["debt_to_income_ratio"] > 43).astype(int)

print(f"Final shape: {df.shape}")
print(f"New features: loan_to_income, high_dti")

Final shape: (2830687, 24)
New features: loan_to_income, high_dti


In [ ]:
print(f"Silver loans: {df.shape}")
print(f"Denial rate: {df['is_denied'].mean():.4f}")
print(f"Originated: {len(df[df['is_denied']==0]):,} | Denied: {len(df[df['is_denied']==1]):,}")

df.to_csv("../A. Data Pipeline/Data/silver/loans_cleaned.csv", index=False)
print("Silver: loans_cleaned.csv saved.")

Silver loans: (2830687, 24)
Denial rate: 0.2582
Originated: 2,099,761 | Denied: 730,926
Silver: loans_cleaned.csv saved.


## **Clean FRED Macro Data**

In [ ]:
import pandas as pd

df_macro = pd.read_csv("../A. Data Pipeline/Data/bronze/fred_macro_raw.csv")
df_macro["date"] = pd.to_datetime(df_macro["date"])
print(f"Raw macro: {df_macro.shape}")
print(f"Date range: {df_macro['date'].min()} → {df_macro['date'].max()}")
df_macro.head()

Raw macro: (2024, 11)
Date range: 2019-01-01 00:00:00 → 2026-08-28 00:00:00


,date,fedfunds,mortgage30us,dgs10,t10y2y,unrate,cpiaucsl,dralacbs,drsfrmacbs,gdp,csushpinsa
0,2019-01-01,2.4,NaN,NaN,NaN,4.0,252.561,1.52,2.69,21111.6,204.167
1,2019-01-02,NaN,NaN,2.66,0.16,NaN,NaN,NaN,NaN,NaN,NaN
2,2019-01-03,NaN,4.51,2.56,0.17,NaN,NaN,NaN,NaN,NaN,NaN
3,2019-01-04,NaN,NaN,2.67,0.17,NaN,NaN,NaN,NaN,NaN,NaN
4,2019-01-07,NaN,NaN,2.70,0.17,NaN,NaN,NaN,NaN,NaN,NaN


In [45]:
df_macro = df_macro.sort_values("date").ffill()
df_macro = df_macro.set_index("date").resample("ME").last().reset_index()
print(f"After resample: {df_macro.shape}")
print(f"Nulls:\n{df_macro.isnull().sum()}")

After resample: (92, 11)
Nulls:
date            0
fedfunds        0
mortgage30us    0
dgs10           0
t10y2y          0
unrate          0
cpiaucsl        0
dralacbs        0
drsfrmacbs      0
gdp             0
csushpinsa      0
dtype: int64


In [46]:
df_macro["mortgage_spread"] = df_macro["mortgage30us"] - df_macro["dgs10"]
df_macro["cpi_yoy"] = df_macro["cpiaucsl"].pct_change(12) * 100
df_macro["home_price_yoy"] = df_macro["csushpinsa"].pct_change(12) * 100
df_macro["activity_year"] = df_macro["date"].dt.year

print("New columns: mortgage_spread, cpi_yoy, home_price_yoy, activity_year")
df_macro.tail()

New columns: mortgage_spread, cpi_yoy, home_price_yoy, activity_year


,date,fedfunds,mortgage30us,dgs10,t10y2y,unrate,cpiaucsl,dralacbs,drsfrmacbs,gdp,csushpinsa,mortgage_spread,cpi_yoy,home_price_yoy,activity_year
87,2026-04-30,3.64,6.30,4.40,0.52,4.3,332.407,1.42,1.86,32486.066,333.109,1.90,3.779246,0.974249,2026
88,2026-05-31,3.63,6.53,4.45,0.47,4.3,333.979,1.42,1.86,32486.066,335.430,2.08,4.166615,1.210861,2026
89,2026-06-30,3.63,6.49,4.44,0.30,4.2,332.568,1.42,1.86,32486.066,336.663,2.05,3.463531,1.525309,2026
90,2026-07-31,3.63,6.66,4.75,0.47,4.1,332.813,1.42,1.86,32486.066,336.663,1.91,3.303856,1.726243,2026
91,2026-08-31,3.63,6.66,4.67,0.39,4.1,332.813,1.42,1.86,32486.066,336.663,1.99,2.945334,2.061462,2026


In [ ]:
df_macro.to_csv("../A. Data Pipeline/Data/silver/macro_cleaned.csv", index=False)
print(f"Silver macro: {df_macro.shape}")
print("Silver: macro_cleaned.csv saved.")

Silver macro: (92, 15)
Silver: macro_cleaned.csv saved.


## **Join Loan + Macro**

In [48]:
macro_yearly = df_macro.groupby("activity_year").agg({
    "fedfunds": "mean",
    "mortgage30us": "mean",
    "dgs10": "mean",
    "unrate": "mean",
    "cpiaucsl": "last",
    "dralacbs": "last",
    "drsfrmacbs": "last",
    "gdp": "last",
    "csushpinsa": "last",
    "mortgage_spread": "mean",
    "cpi_yoy": "last",
    "home_price_yoy": "last",
}).reset_index()

In [ ]:
df_joined = df.merge(macro_yearly, on="activity_year", how="left")
df_joined.to_csv("../A. Data Pipeline/Data/silver/loans_with_macro.csv", index=False)
print(f"Joined dataset: {df_joined.shape}")
print("Silver: loans_with_macro.csv saved.")

Joined dataset: (2830687, 36)
Silver: loans_with_macro.csv saved.
